# Jehovah Witness Data

This notebook loads a local English-Igbo parallel corpus and performs an initial data inspection.

## 1. Package setup

In [26]:
# Run once if the required libraries are not already installed.
# %pip install -q datasets pandas

## 2. Imports and file paths

In [27]:
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from IPython.display import display

# Local dataset files used for this project.
TRAIN_PATH = Path(r"C:\Users\Mr. Paul\Downloads\train.jsonl")
TEST_PATH = Path(r"C:\Users\Mr. Paul\Downloads\test.jsonl")

# Cleaned versions will be saved next to the original files.
CLEANED_TRAIN_PATH = Path(r"C:\Users\Mr. Paul\Downloads\cleaned_train.jsonl")
CLEANED_TEST_PATH = Path(r"C:\Users\Mr. Paul\Downloads\cleaned_test.jsonl")

## 3. Verify file availability

In [28]:
# Checking the paths early makes it easier to catch missing files before loading begins.
print(f"Train file exists: {TRAIN_PATH.exists()} -> {TRAIN_PATH}")
print(f"Test file exists:  {TEST_PATH.exists()} -> {TEST_PATH}")

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError("One or both local JSONL files were not found.")

Train file exists: True -> C:\Users\Mr. Paul\Downloads\train.jsonl
Test file exists:  True -> C:\Users\Mr. Paul\Downloads\test.jsonl


## 4. Load the local dataset

In [29]:
# Reading from local files avoids Hugging Face authentication warnings.
dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_PATH),
        "test": str(TEST_PATH),
    },
)

dataset

DatasetDict({
    train: Dataset({
        features: ['English', 'Igbo'],
        num_rows: 522322
    })
    test: Dataset({
        features: ['English', 'Igbo'],
        num_rows: 3296
    })
})

## 5. Dataset summary

The first step is to confirm the available splits, the number of examples in each split, and the columns included in the corpus.

In [30]:
split_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": split_data.num_rows,
            "columns": ", ".join(split_data.column_names),
        }
        for split_name, split_data in dataset.items()
    ]
)

display(split_summary)

,split,rows,columns
0,train,522322,"English, Igbo"
1,test,3296,"English, Igbo"


## 6. Confirm the schema

In [31]:
train_split = dataset["train"]
test_split = dataset["test"]

print("Train columns:", train_split.column_names)
print("Test columns:", test_split.column_names)

Train columns: ['English', 'Igbo']
Test columns: ['English', 'Igbo']


## 7. Inspect the first few sentence pairs

In [32]:
display(train_split.select(range(5)).to_pandas())

,English,Igbo
0,All this was happening amidst a barrage of reg...,Ihe a niile na-eme n' etiti ọtụtụ twiit nke si...
1,Soon after Jota was denied by the recovering D...,Ozigbo David Sanchez na-agbake agbake gọnahara...
2,Friday’s rout equalled Manchester United’s 9-0...,Asọmụmpi nke Fụraịde nke Manchester United ji ...
3,There were over 70 million Nigerians on the hi...,E nwere ihe karịrị nde ndị Naịjirịa iri asaa n...
4,Diet and Dementia: How you will cure dementia ...,Diet and Dementia: Etu ị ga-esi jiri chocolate...


## 8. Draw a reproducible random sample

A random sample provides a broader sense of the corpus than the first few rows alone.

In [33]:
random_sample = train_split.shuffle(seed=42).select(range(5)).to_pandas()
display(random_sample)

,English,Igbo
0,"Moreover , Paul wrote : “ Do not be anxious ov...","Pọl dere , sị : “ Unu echegbula onwe unu banye..."
1,Table of Contents,Isiokwu Ndị Dị na Magazin A
2,"For instance , if your son recently had a math...","Dị ka ihe atụ , ọ bụrụ na nwa gị nwoke lere ul..."
3,Church Slavic - Xhosa,kwere - Xhosa
4,Present arguments in a clear and logical manner .,"Gwa ya okwu doro anya , nke ga - ekwekwa ya ng..."


## 9. Initial quality check

Before preprocessing, it is useful to measure missing values, blank strings, and approximate sentence length in each split.

In [34]:
def quality_report(split_name, split_data):
    missing_english = 0
    missing_igbo = 0
    empty_english = 0
    empty_igbo = 0
    total_english_words = 0
    total_igbo_words = 0

    for row in split_data:
        english = row["English"]
        igbo = row["Igbo"]

        if english is None:
            missing_english += 1
        elif not english.strip():
            empty_english += 1
        else:
            total_english_words += len(english.split())

        if igbo is None:
            missing_igbo += 1
        elif not igbo.strip():
            empty_igbo += 1
        else:
            total_igbo_words += len(igbo.split())

    row_count = split_data.num_rows
    non_empty_english = row_count - missing_english - empty_english
    non_empty_igbo = row_count - missing_igbo - empty_igbo

    return {
        "split": split_name,
        "rows": row_count,
        "missing_english": missing_english,
        "missing_igbo": missing_igbo,
        "empty_english": empty_english,
        "empty_igbo": empty_igbo,
        "avg_english_words": round(total_english_words / non_empty_english, 2) if non_empty_english else 0,
        "avg_igbo_words": round(total_igbo_words / non_empty_igbo, 2) if non_empty_igbo else 0,
    }

quality_df = pd.DataFrame(
    [
        quality_report("train", train_split),
        quality_report("test", test_split),
    ]
)

display(quality_df)

,split,rows,missing_english,missing_igbo,empty_english,empty_igbo,avg_english_words,avg_igbo_words
0,train,522322,0,0,13024,3222,17.57,21.55
1,test,3296,0,0,1,15,23.39,22.31


## 10. Remove blank sentence pairs

For the next stages of the project, it is reasonable to keep only rows where both the English and Igbo fields contain usable text.

In [35]:
clean_train = train_split.filter(
    lambda row: bool(row["English"] and row["English"].strip() and row["Igbo"] and row["Igbo"].strip())
)

clean_test = test_split.filter(
    lambda row: bool(row["English"] and row["English"].strip() and row["Igbo"] and row["Igbo"].strip())
)

pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows_before": [train_split.num_rows, test_split.num_rows],
        "rows_after": [clean_train.num_rows, clean_test.num_rows],
    }
)

,split,rows_before,rows_after
0,train,522322,506078
1,test,3296,3281


## 11. Inspect the cleaned data

In [36]:
display(clean_train.select(range(5)).to_pandas())

,English,Igbo
0,All this was happening amidst a barrage of reg...,Ihe a niile na-eme n' etiti ọtụtụ twiit nke si...
1,Soon after Jota was denied by the recovering D...,Ozigbo David Sanchez na-agbake agbake gọnahara...
2,Friday’s rout equalled Manchester United’s 9-0...,Asọmụmpi nke Fụraịde nke Manchester United ji ...
3,There were over 70 million Nigerians on the hi...,E nwere ihe karịrị nde ndị Naịjirịa iri asaa n...
4,Diet and Dementia: How you will cure dementia ...,Diet and Dementia: Etu ị ga-esi jiri chocolate...


## 12. Save cleaned files

The cleaned train and test splits can be exported as JSONL files next to the original downloads.

In [37]:
# Export the cleaned splits as JSONL files in the Downloads folder.
clean_train.to_json(str(CLEANED_TRAIN_PATH), force_ascii=False)
clean_test.to_json(str(CLEANED_TEST_PATH), force_ascii=False)

print(f"Saved cleaned train split to: {CLEANED_TRAIN_PATH}")
print(f"Saved cleaned test split to:  {CLEANED_TEST_PATH}")

Creating json from Arrow format: 100%|██████████| 4/4 [00:00<00:00, 126.78ba/s]

Saved cleaned train split to: C:\Users\Mr. Paul\Downloads\cleaned_train.jsonl
Saved cleaned test split to:  C:\Users\Mr. Paul\Downloads\cleaned_test.jsonl
